# Sujet 2 · Des profils d'apprenants : le clustering d'un questionnaire

**Cours « Intelligence artificielle »** · Master 2 (ANG, LGC, DID) · Université Yahia Farès, Médéa
**Chapitre 3** (apprentissage non supervisé : les k-moyennes) · **Mini-projet, sujet 2** (clustering d'enquêtes) · **Durée :** environ 1 h 30

**Ce que vous allez faire**
- fabriquer un questionnaire fictif rempli par 120 apprenants de français (motivation, lecture, anxiété, usage de l'IA…), puis le standardiser ;
- choisir le nombre de groupes avec la méthode du coude et lancer l'algorithme des k-moyennes (*k-means*) ;
- décrire et nommer les profils d'apprenants obtenus, puis vous demander à quoi ils peuvent servir en classe, et avec quelles précautions.

**Mode d'emploi**
Exécutez les cellules de haut en bas : cliquez sur le bouton ▶ à gauche de chaque cellule, ou appuyez sur Maj+Entrée. Vous n'avez jamais à écrire de code. Modifiez seulement les champs des cellules intitulées « À MODIFIER ». Dans Colab, le menu « Exécution → Tout exécuter » lance tout le carnet d'un coup.

Avant chaque cellule de code, une question vous demande de **prédire** le résultat : notez votre réponse, exécutez, puis comparez avec la rubrique « Ce que vous devez voir ». Python écrit les nombres décimaux avec un point : 0.39 se lit 0,39.

> **Données fictives.** Toutes les réponses de ce carnet sont inventées par un programme, à partir d'une graine fixe. Aucun apprenant réel n'a été interrogé : ne citez jamais ces chiffres comme un résultat d'enquête.

## Étape 1 · Charger les outils

Cette cellule charge les bibliothèques du cours : NumPy (calcul), pandas (tableaux), matplotlib (graphiques) et scikit-learn (apprentissage automatique). Elle fixe aussi l'ordre des couleurs des groupes.

**Avant d'exécuter :** cette cellule va-t-elle afficher un tableau, un graphique, ou seulement un court message ?

In [ ]:
#@title Charger les outils (bibliothèques)
import numpy as np                     # calcul sur des séries de nombres
import pandas as pd                    # tableaux de données
import matplotlib.pyplot as plt        # graphiques
from matplotlib.colors import LinearSegmentedColormap   # dégradé de couleurs
from sklearn.preprocessing import StandardScaler        # standardisation
from sklearn.cluster import KMeans                      # algorithme des k-moyennes
from sklearn.metrics import silhouette_score            # score de silhouette

# Couleurs et formes des groupes, toujours dans le même ordre
COULEURS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
FORMES = ["o", "s", "^", "D", "v", "P", "X", "*"]
print("Outils chargés.")

**Ce que vous devez voir :** un seul message, `Outils chargés.` Le chargement ne produit rien d'autre : il prépare le travail.

**Question :** retrouvez dans la cellule le nom de l'outil qui fabriquera les groupes. Quel mot du cours reconnaissez-vous dans son nom ?

## Étape 2 · Le questionnaire et ses profils cachés

Notre questionnaire fictif pose six questions fermées à des étudiants de licence qui apprennent le français. Voici son **dictionnaire des variables**, le document que le mini-projet vous demande de rédiger pour vos propres données :

| Variable | Question posée | Codage de la réponse |
|---|---|---|
| `motivation` | « Je suis motivé(e) pour apprendre le français. » | 1 (pas du tout d'accord) à 5 (tout à fait d'accord) |
| `lecture` | « Combien d'heures par semaine lisez-vous en français, hors cours ? » | nombre d'heures |
| `anxiete` | « Je suis anxieux (anxieuse) quand je dois parler français en classe. » | 1 à 5 |
| `ia` | « J'utilise un outil d'IA (ChatGPT, traducteur automatique) pour faire mes devoirs. » | 1 (jamais) à 5 (toujours) |
| `oral` | « Je prends la parole en classe. » | 1 (jamais) à 5 (très souvent) |
| `ecrans` | « Combien d'heures par semaine regardez-vous des séries ou des vidéos en français ? » | nombre d'heures |

La question sur l'anxiété s'inspire d'une échelle réelle de la didactique des langues, la *Foreign Language Classroom Anxiety Scale* de Horwitz, Horwitz et Cope (1986), qui compte 33 affirmations.

La cellule suivante choisit la **graine** du hasard (le même nombre donne toujours les mêmes réponses) et décrit quatre **profils cachés**, A, B, C et D : pour chacun, le nombre d'apprenants et la réponse moyenne à chaque question. Le programme tirera ensuite des réponses autour de ces moyennes. Nous ne vous disons pas encore ce que représentent ces profils : l'algorithme, lui, ne les verra jamais.

**Avant d'exécuter :** en lisant les moyennes, quel profil vous semble correspondre à de grands lecteurs ? Lequel à des apprenants très anxieux ?

In [ ]:
#@title À MODIFIER : la graine du hasard, puis exécutez (les quatre profils cachés)
graine = 2026 #@param {type:"integer"}
rng = np.random.default_rng(graine)    # générateur de hasard reproductible
colonnes = ["motivation", "lecture", "anxiete", "ia", "oral", "ecrans"]
echelles = ["motivation", "anxiete", "ia", "oral"]     # réponses de 1 à 5
heures = ["lecture", "ecrans"]                         # heures par semaine
profils_caches = {   # profil : (nombre d'apprenants, réponse moyenne à chaque question)
    "A": (30, [4.5, 7.0, 2.0, 1.8, 4.0, 5]),
    "B": (35, [3.8, 3.5, 4.4, 2.3, 1.6, 4]),
    "C": (35, [3.6, 1.5, 2.2, 4.2, 3.8, 12]),
    "D": (20, [1.7, 0.5, 3.2, 4.4, 1.8, 2]),
}
dispersion = [0.7, 1.5, 0.7, 0.7, 0.7, 3]   # écart habituel autour de chaque moyenne
print(pd.DataFrame({nom: moy for nom, (n, moy) in profils_caches.items()}, index=colonnes))

**Ce que vous devez voir :** un tableau de 6 lignes (les questions) et 4 colonnes (les profils A, B, C, D). Le profil A lit 7 heures par semaine, le profil B a l'anxiété la plus forte (4.4) et la prise de parole la plus faible (1.6), le profil C passe 12 heures devant des vidéos, le profil D a la motivation la plus basse (1.7).

**Question :** ces moyennes sont des choix de l'auteur du carnet. Dans une vraie enquête, qui fixe les profils ? Peut-on même savoir s'il en existe ?

Cette cellule tire au hasard les 120 réponses autour des moyennes des profils, les arrondit comme sur un vrai questionnaire (des entiers de 1 à 5 pour les échelles, des heures entières), mélange les lignes, puis **met de côté** la colonne du profil caché : l'algorithme ne travaillera que sur les six réponses.

**Avant d'exécuter :** combien de lignes et combien de colonnes aura le tableau des réponses ?

In [ ]:
#@title Tirer au hasard les 120 réponses fictives
morceaux = []
for nom, (effectif, moyennes) in profils_caches.items():
    bloc = pd.DataFrame(rng.normal(moyennes, dispersion, size=(effectif, 6)), columns=colonnes)
    bloc["profil_cache"] = nom
    morceaux.append(bloc)
reponses = pd.concat(morceaux, ignore_index=True)
# Arrondir comme sur un vrai questionnaire : entiers de 1 à 5, heures entières
reponses[echelles] = reponses[echelles].round().clip(1, 5)
reponses[heures] = reponses[heures].round().clip(0, None)
reponses[colonnes] = reponses[colonnes].astype(int)
reponses = reponses.sample(frac=1, random_state=graine).reset_index(drop=True)  # mélanger
profil_cache = reponses.pop("profil_cache")      # mettre de côté le profil caché
print("Lignes, colonnes :", reponses.shape)
reponses.head(8)

**Ce que vous devez voir :** `Lignes, colonnes : (120, 6)`, puis les 8 premiers apprenants. Le premier (ligne 0) a répondu 4, 3, 4, 2, 2, 2 : motivé, 3 heures de lecture, assez anxieux, peu d'IA, prend peu la parole, 2 heures de vidéos. Le troisième (ligne 2) lit 7 heures par semaine avec une anxiété de 1.

**Question :** en lisant seulement ces 8 lignes, pouvez-vous deviner le profil caché de chaque apprenant ? Qu'est-ce qui rend la tâche difficile ?

## Étape 3 · Explorer les réponses

Avant tout calcul savant, on regarde les données : moyenne, écart type (la dispersion autour de la moyenne), minimum et maximum de chaque question, puis un histogramme par question.

**Avant d'exécuter :** quelle question aura les plus grands nombres ? Laquelle aura l'écart type le plus élevé ?

In [ ]:
#@title Résumer et dessiner les réponses à chaque question
resume = reponses.describe().round(1)
print(resume.loc[["mean", "std", "min", "max"]])   # moyenne, écart type, min, max
reponses.hist(figsize=(10, 6), bins=12, color=COULEURS[0], edgecolor="white", grid=False)
plt.suptitle("Répartition des réponses (données fictives, 120 apprenants)")
plt.tight_layout()
plt.show()

**Ce que vous devez voir :** un petit tableau (lignes `mean`, `std`, `min`, `max`) et six histogrammes. La question `ecrans` a les plus grands nombres (de 0 à 17 heures, moyenne 6.7) et l'écart type le plus élevé (4.2) ; `lecture` va de 0 à 12 heures (écart type 2.7) ; les quatre échelles restent entre 1 et 5, avec un écart type proche de 1.1 à 1.3.

**Question :** si l'on mesurait la « distance » entre deux apprenants avec ces nombres bruts, quelle question compterait le plus ? Est-ce juste ?

## Étape 4 · Standardiser

Les k-moyennes regroupent les apprenants **proches** les uns des autres. Pour que chaque question pèse autant, on la standardise : on retire la moyenne et on divise par l'écart type. Une réponse standardisée dit de combien d'écarts types l'apprenant s'éloigne de la moyenne : 0 = dans la moyenne, +1 = nettement au-dessus, −1 = nettement en dessous.

valeur standardisée = (réponse − moyenne) / écart type

**Avant d'exécuter :** le premier apprenant lit 3 heures par semaine, pour une moyenne de 3,4 heures. Sa valeur standardisée pour `lecture` sera-t-elle positive ou négative ? Proche de 0 ou de 1 ?

In [ ]:
#@title Standardiser : mettre toutes les questions à la même échelle
X = StandardScaler().fit_transform(reponses[colonnes])
X_tableau = pd.DataFrame(X, columns=colonnes).round(2)
print("Trois apprenants AVANT (réponses brutes) :")
print(reponses[colonnes].head(3))
print()
print("Les mêmes APRÈS standardisation :")
print(X_tableau.head(3))
print()
print("Écart type de chaque question après :", X_tableau.std(ddof=0).round(1).tolist())

**Ce que vous devez voir :** pour le premier apprenant, `lecture` passe de 3 à −0.14 (un peu sous la moyenne), `anxiete` de 4 à 0.81 et `ecrans` de 2 à −1.12 (nettement en dessous). Toutes les questions ont maintenant un écart type de 1.0. Vérifiez à la main : (3 − 3,38) / 2,72 ≈ −0,14.

**Question :** l'apprenant de la ligne 2 lit 7 heures par semaine. Pourquoi sa valeur standardisée (1.33) est-elle bien plus petite que 7 ?

## Étape 5 · Combien de groupes ? La méthode du coude

L'algorithme des k-moyennes demande qu'on lui donne k, le nombre de groupes. Pour choisir, on l'essaie avec k = 1, 2, … 8 et on note l'**inertie** : la somme des carrés des distances entre chaque apprenant et le centre de son groupe. Elle baisse toujours quand k augmente ; on retient la valeur de k à partir de laquelle elle ne baisse presque plus (le « coude » de la courbe). On calcule aussi le **score de silhouette** (de −1 à 1) : plus il est élevé, mieux les groupes sont séparés.

**Avant d'exécuter :** où pensez-vous voir le coude ? Le score de silhouette sera-t-il le plus haut pour le même k ?

In [ ]:
#@title Méthode du coude : essayer de k = 1 à k = 8 groupes
resultats = []
for k_essai in range(1, 9):
    km = KMeans(n_clusters=k_essai, n_init=10, random_state=0).fit(X)
    # La silhouette n'existe pas pour un seul groupe
    silhouette = silhouette_score(X, km.labels_) if k_essai > 1 else np.nan
    resultats.append({"k": k_essai, "inertie": round(km.inertia_, 1), "silhouette": round(silhouette, 3)})
coude = pd.DataFrame(resultats)
print(coude.to_string(index=False))
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(coude["k"], coude["inertie"], marker="o", color=COULEURS[0], linewidth=2)
ax.set_xlabel("Nombre de groupes k")
ax.set_ylabel("Inertie (dispersion dans les groupes)")
ax.set_title("Méthode du coude (données fictives)")
plt.show()

**Ce que vous devez voir :** l'inertie passe de 720 (k = 1) à 502,1, puis 348,3, puis **246,5 pour k = 4** ; ensuite elle ne baisse plus que d'une vingtaine de points à chaque groupe ajouté (226,2 ; 207,9 ; 197,4 ; 187,1). Le coude est à k = 4, et le score de silhouette est aussi le plus haut pour k = 4 (0,387, contre 0,342 pour k = 3 et 0,324 pour k = 5).

**Question :** pourquoi ne pas simplement choisir k = 8, qui a l'inertie la plus basse ? Que deviendrait l'inertie avec 120 groupes, un par apprenant ?

## Étape 6 · Lancer les k-moyennes

L'algorithme place k centres, affecte chaque apprenant au centre le plus proche, recalcule chaque centre comme la moyenne de son groupe, et recommence jusqu'à ce que plus personne ne change de groupe. `depart` fixe le hasard du placement initial des centres ; `n_init=10` relance l'algorithme depuis dix départs et garde la meilleure solution.

**Avant d'exécuter :** avec k = 4, les quatre groupes auront-ils la même taille ? Rappelez-vous les effectifs des profils cachés (30, 35, 35, 20).

In [ ]:
#@title À MODIFIER : le nombre de groupes k, puis exécutez (k-moyennes)
k = 4 #@param {type:"slider", min:2, max:8, step:1}
depart = 0 #@param {type:"integer"}
modele = KMeans(n_clusters=k, n_init=10, random_state=depart)
reponses["groupe"] = modele.fit_predict(X)   # un numéro de groupe par apprenant
print("Nombre d'apprenants dans chaque groupe :")
print(reponses["groupe"].value_counts().sort_index().to_string())

**Ce que vous devez voir :** quatre groupes numérotés de 0 à 3, de tailles différentes : 32, 21, 32 et 35 apprenants (total 120). Les tailles ressemblent à celles des profils cachés sans être identiques.

**Question :** le groupe 0 est-il « meilleur » ou « premier » par rapport au groupe 3 ? Que signifient, selon vous, ces numéros ?

## Étape 7 · Décrire les profils

Les numéros de groupe ne veulent rien dire en eux-mêmes : seul le **profil moyen** de chaque groupe permet de le décrire. Cette cellule calcule la moyenne de chaque question dans chaque groupe, dans les unités du questionnaire (échelle de 1 à 5, heures).

**Avant d'exécuter :** quel groupe aura, selon vous, la moyenne de `lecture` la plus haute ?

In [ ]:
#@title Calculer le profil moyen de chaque groupe
profils = reponses.groupby("groupe")[colonnes].mean().round(1)
profils["effectif"] = reponses["groupe"].value_counts().sort_index()
profils

**Ce que vous devez voir :** un tableau de 4 lignes.
- Groupe 0 (32) : motivation 4.4, lecture 6.7 h, anxiété 2.1, IA 1.9, oral 4.0, écrans 6.3 h.
- Groupe 1 (21) : motivation 1.6, lecture 1.4 h, anxiété 3.3, IA 4.0, oral 2.0, écrans 2.6 h.
- Groupe 2 (32) : motivation 3.5, lecture 1.3 h, anxiété 2.6, IA 4.4, oral 3.9, écrans 11.6 h.
- Groupe 3 (35) : motivation 3.9, lecture 3.5 h, anxiété 4.3, IA 2.2, oral 1.7, écrans 4.9 h.

**Question :** les groupes 1 et 2 lisent aussi peu l'un que l'autre. Sur quelles questions se distinguent-ils nettement ?

Un tableau de chiffres se lit mal. Cette cellule dessine le **portrait** des groupes : chaque case donne l'écart du centre du groupe à la moyenne de tous les apprenants, en écarts types (les valeurs standardisées de l'étape 4). Rouge : au-dessus de la moyenne ; bleu : en dessous ; gris : dans la moyenne.

**Avant d'exécuter :** quelle case sera la plus rouge ? Laquelle sera la plus bleue ?

In [ ]:
#@title Dessiner le portrait des groupes (écarts à la moyenne)
centres = pd.DataFrame(modele.cluster_centers_, columns=colonnes)   # centres standardisés
bleu_rouge = LinearSegmentedColormap.from_list("ecart", ["#2a78d6", "#f0efec", "#e34948"])
fig, ax = plt.subplots(figsize=(8, 1 + 0.7 * k))
image = ax.imshow(centres, cmap=bleu_rouge, vmin=-2, vmax=2, aspect="auto")
ax.set_xticks(range(len(colonnes)), colonnes)
ax.set_yticks(range(k), [f"Groupe {g}" for g in range(k)])
for g in range(k):
    for j in range(len(colonnes)):
        ax.text(j, g, f"{centres.iloc[g, j]:+.1f}", ha="center", va="center")
fig.colorbar(image, label="écart à la moyenne (en écarts types)")
ax.set_title("Portrait des groupes : rouge = plus que la moyenne, bleu = moins")
plt.show()

**Ce que vous devez voir :** une grille de 4 lignes et 6 colonnes. Les cases les plus rouges : `ecrans` du groupe 2 (+1.2) et `lecture` du groupe 0 (+1.2). Les plus bleues : `motivation` du groupe 1 (−1.7), puis `ecrans` du groupe 1 et `oral` du groupe 3 (−1.0). Le groupe 3 a aussi une anxiété nettement au-dessus de la moyenne (+1.1).

**Question :** relisez la ligne du groupe 1. Quelle case vous paraît la plus importante pour un enseignant, et pourquoi ?

Cette cellule traduit le portrait en mots : pour chaque groupe, elle liste les questions où le centre du groupe s'écarte de la moyenne de plus d'un certain **seuil** (en écarts types). Vous pouvez changer ce seuil avec le curseur.

**Avant d'exécuter :** avec un seuil de 0,5, combien de questions seront citées pour le groupe 0 ?

In [ ]:
#@title À MODIFIER : le seuil, puis exécutez (description des groupes en mots)
seuil = 0.5 #@param {type:"slider", min:0.25, max:1.5, step:0.25}
for g in range(k):
    centre = centres.iloc[g]
    plus = [q for q in colonnes if centre[q] > seuil]
    moins = [q for q in colonnes if centre[q] < -seuil]
    print(f"Groupe {g} ({profils.loc[g, 'effectif']} apprenants)")
    print("   nettement plus que la moyenne  :", ", ".join(plus) or "rien")
    print("   nettement moins que la moyenne :", ", ".join(moins) or "rien")

**Ce que vous devez voir :**
- Groupe 0 (32 apprenants) : plus que la moyenne : motivation, lecture, oral ; moins : anxiete, ia.
- Groupe 1 (21) : plus : ia ; moins : motivation, lecture, oral, ecrans.
- Groupe 2 (32) : plus : ia, oral, ecrans ; moins : lecture.
- Groupe 3 (35) : plus : anxiete ; moins : ia, oral.

**Question :** montez le seuil à 1,0 et relancez. Que reste-t-il pour chaque groupe ? Ce qui reste est-il le cœur du profil ?

## Étape 8 · Nommer les profils

C'est maintenant au chercheur, c'est-à-dire à vous, de **nommer** les groupes. Un bon nom résume le profil sans le caricaturer. Les noms proposés ci-dessous correspondent aux groupes obtenus avec les réglages par défaut (graine 2026, k = 4, départ 0) ; si vous avez changé un réglage, relisez les profils et renommez.

**Avant d'exécuter :** écrivez sur papier votre propre nom pour chacun des quatre groupes, puis comparez avec les noms proposés dans les champs. Remplacez-les par les vôtres si vous les préférez.

In [ ]:
#@title À MODIFIER : donnez un nom à chaque groupe, puis exécutez
nom_groupe_0 = "Lecteurs autonomes" #@param {type:"string"}
nom_groupe_1 = "Apprenants en retrait" #@param {type:"string"}
nom_groupe_2 = "Connectés (écrans, IA, oral)" #@param {type:"string"}
nom_groupe_3 = "Motivés mais anxieux" #@param {type:"string"}
noms = {0: nom_groupe_0, 1: nom_groupe_1, 2: nom_groupe_2, 3: nom_groupe_3}
# Un groupe sans nom (si k est plus grand que 4) garde son numéro
reponses["profil"] = [noms.get(g, f"Groupe {g}") for g in reponses["groupe"]]
print(reponses["profil"].value_counts().to_string())

**Ce que vous devez voir :** les quatre noms avec leur effectif, du plus grand au plus petit : Motivés mais anxieux 35, Lecteurs autonomes 32, Connectés (écrans, IA, oral) 32, Apprenants en retrait 21.

Nommer, c'est interpréter. Comme les idéaux-types de Max Weber, ces profils sont des constructions qui simplifient la réalité pour la rendre lisible. Un nom trop sévère (« paresseux », « tricheurs à l'IA ») colle à des personnes réelles et peut orienter le regard de l'enseignant.

**Question :** le nom « Apprenants en retrait » décrit-il ce que font ces apprenants ou ce qu'ils sont ? Proposez un nom plus prudent, ou au contraire plus précis.

## Étape 9 · Voir les groupes sur un graphique

Chaque apprenant a six réponses : on ne peut pas dessiner six dimensions. On choisit donc **deux questions** et on place chaque apprenant sur un plan, avec la couleur et la forme de son groupe. Un léger décalage au hasard évite que les points identiques se cachent les uns les autres.

**Avant d'exécuter :** avec `lecture` en abscisse et `ia` en ordonnée, les « Lecteurs autonomes » seront-ils en haut à gauche ou en bas à droite ?

In [ ]:
#@title À MODIFIER : deux questions à croiser, puis exécutez (nuage de points)
axe_x = "lecture" #@param ["motivation", "lecture", "anxiete", "ia", "oral", "ecrans"] {allow-input: true}
axe_y = "ia" #@param ["motivation", "lecture", "anxiete", "ia", "oral", "ecrans"] {allow-input: true}
fig, ax = plt.subplots(figsize=(7, 5))
for g, groupe in reponses.groupby("groupe"):
    decalage = rng.uniform(-0.25, 0.25, size=(len(groupe), 2))   # léger décalage au hasard
    ax.scatter(groupe[axe_x] + decalage[:, 0], groupe[axe_y] + decalage[:, 1], s=40, alpha=0.8,
               color=COULEURS[g], marker=FORMES[g], label=groupe["profil"].iloc[0])
ax.set_xlabel(axe_x)
ax.set_ylabel(axe_y)
ax.set_title("Chaque point est un apprenant fictif")
ax.legend(title="Profil")
plt.show()

**Ce que vous devez voir :** 120 points. Les « Lecteurs autonomes » (ronds bleus) sont en bas à droite : beaucoup de lecture, peu d'IA. Les « Motivés mais anxieux » (losanges jaunes) occupent le bas du milieu : de 1 à 6 heures de lecture, peu d'IA. Les « Apprenants en retrait » (carrés orange) et les « Connectés » (triangles verts) se mélangent en haut à gauche : peu de lecture, beaucoup d'IA.

**Question :** choisissez deux autres questions qui séparent bien les « Apprenants en retrait » des « Connectés » (indice : regardez le portrait de l'étape 7), puis relancez la cellule. Lesquelles avez-vous choisies ?

## Étape 10 · Révélation : l'algorithme a-t-il retrouvé les profils cachés ?

Nos données étant fictives, nous connaissons la « bonne réponse » : le profil caché de chaque apprenant, mis de côté à l'étape 2. Ce tableau croise vos groupes (lignes) et les profils cachés (colonnes). Chaque case compte les apprenants concernés.

**Avant d'exécuter :** combien d'apprenants, sur 120, seront selon vous dans un groupe qui ne correspond pas à leur profil caché ?

In [ ]:
#@title Comparer les groupes trouvés et les profils cachés
pd.crosstab(reponses["profil"], profil_cache, colnames=["profil caché"])

**Ce que vous devez voir :** presque tout est sur une seule case par ligne. Lecteurs autonomes : 30 A et 2 C ; Apprenants en retrait : 20 D et 1 B ; Connectés : 32 C ; Motivés mais anxieux : 34 B et 1 C. Seuls 4 apprenants sur 120 sont « mal placés ».

Voici ce que l'auteur du carnet avait voulu fabriquer : **A** = grands lecteurs autonomes, **B** = apprenants motivés mais anxieux à l'oral, **C** = apprenants tournés vers les écrans et l'IA, qui parlent volontiers, **D** = apprenants démotivés.

**Question :** dans une vraie enquête, cette colonne « profil caché » n'existe pas. Comment pourriez-vous alors vérifier que vos groupes ont un sens (relire des questionnaires, interroger des apprenants, relancer avec un autre départ…) ?

## À vous

Modifiez seulement les champs, puis relancez les cellules à partir de celle que vous avez changée (dans Colab : « Exécution → Exécuter les cellules suivantes »).
1. **Étape 6 :** mettez k = 3, puis k = 5. Quels profils fusionnent avec k = 3 ? Quel profil se coupe en deux avec k = 5 ? (Pensez à relire l'étape 7 : les noms de l'étape 8 ne correspondent plus.)
2. **Étape 6, stabilité :** remettez k = 4 et changez `depart` (1, puis 2). Les numéros de groupes changent-ils ? Les profils restent-ils reconnaissables ?
3. **Étape 2 :** changez la graine (par exemple 7) et relancez tout le carnet. Le coude est-il toujours à k = 4 ?

## Pour le mini-projet : vos propres données

Pour votre mini-projet, remplacez les réponses fictives par celles de **votre** questionnaire :
- préparez un fichier CSV (dans un tableur : « Enregistrer sous… → CSV ») avec **une ligne par répondant** et **une colonne par question**, chaque réponse codée par un nombre ;
- questionnaire anonyme, rempli avec le consentement des répondants ; retirez noms et identifiants (en Algérie, la loi n° 18-07 du 10 juin 2018 encadre les données personnelles) ;
- cochez la case ci-dessous et exécutez la cellule : dans Colab, une fenêtre vous demande le fichier ; dans Jupyter, placez le fichier à côté du carnet sous le nom `mon_questionnaire.csv` ;
- la cellule garde les colonnes chiffrées et retire les questionnaires incomplets (dites-le dans votre rapport) ; relancez ensuite les étapes 4 à 9, en tapant à l'étape 9 le nom de deux de vos questions. Sautez l'étape 10 : vos données n'ont pas de profil caché.

**Avant d'exécuter :** si vous laissez la case décochée, que va faire la cellule ?

In [ ]:
#@title À MODIFIER : charger votre propre questionnaire (fichier CSV)
charger_mon_fichier = False #@param {type:"boolean"}
if charger_mon_fichier:
    try:
        from google.colab import files       # n'existe que dans Colab
        envoi = files.upload()               # fenêtre pour choisir le fichier
        nom_fichier = list(envoi.keys())[0]
    except ImportError:                      # Jupyter : fichier placé à côté du carnet
        nom_fichier = "mon_questionnaire.csv"
    mes_reponses = pd.read_csv(nom_fichier)
    colonnes = list(mes_reponses.select_dtypes("number").columns)   # questions chiffrées
    reponses = mes_reponses[colonnes].dropna().reset_index(drop=True)
    print(len(reponses), "réponses complètes ; questions :", colonnes)
else:
    print("Case non cochée : le carnet garde les données fictives.")

**Ce que vous devez voir :** avec la case décochée, `Case non cochée : le carnet garde les données fictives.` Avec votre fichier, le nombre de réponses complètes et la liste de vos questions chiffrées.

**Question :** une question ouverte (« Pourquoi apprenez-vous le français ? ») peut-elle entrer telle quelle dans les k-moyennes ? Comment pourriez-vous la coder en nombres ?

## À rendre

Répondez par écrit, en français, en quelques lignes pour chaque question.
1. **Choisir k.** Justifiez le nombre de groupes retenu à l'aide de la courbe du coude et du score de silhouette. Que perdez-vous avec k = 3 ? Que gagnez-vous (ou non) avec k = 5 ?
2. **Décrire et nommer.** Décrivez les quatre profils avec les chiffres du tableau de l'étape 7, puis justifiez le nom que vous donnez à chacun. Quel nom avez-vous modifié, et pourquoi ?
3. **En classe (didactique).** Pour deux des profils, proposez une activité ou un dispositif adapté (pédagogie différenciée). Puis indiquez une limite de ce classement : taille de l'échantillon, réponses déclaratives, risque d'enfermer un apprenant dans une étiquette…

## Ce qu'il faut retenir

- Le **clustering** forme des groupes sans étiquettes données à l'avance : c'est de l'apprentissage **non supervisé**. Avant les k-moyennes, on **standardise**, sinon la question aux plus grands nombres décide seule.
- La **méthode du coude** (inertie) et le **score de silhouette** aident à choisir k, mais c'est le chercheur qui décide, puis qui décrit les groupes par leurs moyennes.
- Les numéros de groupes sont arbitraires ; les **profils** sont des constructions à nommer avec prudence et à confronter au terrain.

**Leçons du site :** [Mini-projet, sujet 2 : clustering d'enquêtes](https://progremer04.github.io/canva-au-for-ang-/#mp-sujet-2) · [Chapitre 3 : le regroupement en clusters (k-means)](https://progremer04.github.io/canva-au-for-ang-/#c3-algorithmes)